# AOOP ARMONYX — Notebook 2: Anime Image Worker

## Endpoints

```
GET  /health
POST /generate-scene          <- async 202 — gera 1 imagem
POST /generate-images         <- async 202 — gera todas as cenas do storyboard
GET  /status/{job_id}
GET  /result/{job_id}/scene/{index}   <- PNG binário
GET  /result/{job_id}/images          <- JSON com todas as imagens em base64
```

## Notas
- Anything V5 (stablediffusionapi/anything-v5) corre na GPU 0 em float16.
- CPU offload activo — cada componente carrega para GPU só quando necessário.
- Secrets via Kaggle Secrets (NGROK_TOKEN, NGROK_DOMAIN, HF_TOKEN, WORKER_API_KEY).
- Sem escrita em disco — imagens geradas em memória e enviadas via StreamingResponse.

## 1 - Instalar dependências

In [ ]:
import os, sys, subprocess

def run(cmd):
    print(">>>", " ".join(cmd))
    result = subprocess.run(cmd)
    if result.returncode != 0:
        raise RuntimeError(f"Comando falhou: {' '.join(cmd)}")

subprocess.run(["apt-get", "update", "-qq"], check=False)
subprocess.run(["apt-get", "install", "-y", "-qq", "ffmpeg", "curl", "libgl1"], check=True)

run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "pip", "setuptools", "wheel"])

# Remove os conflitos principais
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y",
                "flax", "jax", "jaxlib", "peft"], check=False)

# Actualiza transformers para versão que tem EncoderDecoderCache
run([sys.executable, "-m", "pip", "install", "-q", "--upgrade",
     "transformers",
     "diffusers",
     "accelerate",
     "safetensors",
     "fastapi", "uvicorn[standard]", "pyngrok",
     "requests", "httpx", "pydantic", "Pillow"])

run([sys.executable, "-m", "pip", "install", "-q",
     "Pillow==10.4.0",
     "--force-reinstall"])

print("Dependências instaladas correctamente.")

## 2 - Imports e configuração

In [ ]:
import io, os, re, json, time, uuid, base64, threading
from datetime import datetime, timedelta
from pathlib import Path
from typing import Optional, Dict, Any, List

import requests
import torch
from PIL import Image

from fastapi import FastAPI, HTTPException, Header, BackgroundTasks
from fastapi.responses import StreamingResponse
from pydantic import BaseModel, Field
from pyngrok import ngrok, conf
from diffusers import StableDiffusionPipeline, EulerDiscreteScheduler
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

# ── Secrets
_secrets       = UserSecretsClient()
NGROK_TOKEN    = _secrets.get_secret("NGROK_TOKEN")
NGROK_DOMAIN   = _secrets.get_secret("NGROK_DOMAIN")
HF_TOKEN       = _secrets.get_secret("HF_TOKEN")
WORKER_API_KEY = _secrets.get_secret("WORKER_API_KEY")
login(token=HF_TOKEN)
print("Hugging Face autenticado.")

# ── Configuração
SDXL_MODEL_NAME = "stablediffusionapi/anything-v5"
SDXL_DEVICE      = "cuda:0"
PORT             = 8000
JOB_TTL_MINUTES  = 60

# ── Defaults de geração
DEFAULT_WIDTH        = 512
DEFAULT_HEIGHT       = 288  # 16:9 para videoclip
DEFAULT_STEPS        = 25
DEFAULT_GUIDANCE     = 7.5
DEFAULT_ANIME_SUFFIX = (
    "anime style, detailed, cinematic lighting, vibrant colors, "
    "high quality, sharp focus, dramatic composition, "
    "masterpiece, best quality"
)
NEGATIVE_PROMPT = (
    "lowres, bad anatomy, bad hands, text, error, missing fingers, "
    "extra digit, fewer digits, cropped, worst quality, low quality, "
    "normal quality, jpeg artifacts, signature, watermark, username, blurry, "
    "realistic, photograph, 3d render, western style"
)

print(f"PyTorch: {torch.__version__}")
print(f"CUDA disponível: {torch.cuda.is_available()}")
print(f"GPUs disponíveis: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f"GPU {i}: {props.name} — {props.total_memory // 1024**2} MB")

## 3 - Carregar SDXL na GPU 0

In [ ]:
print(f"A carregar {SDXL_MODEL_NAME} em {SDXL_DEVICE}...")
print("(pode demorar 1-2 min no primeiro carregamento)")

pipeline = StableDiffusionPipeline.from_pretrained(
    SDXL_MODEL_NAME,
    torch_dtype=torch.float16,
    use_safetensors=False,
)

pipeline.scheduler = EulerDiscreteScheduler.from_config(pipeline.scheduler.config)
pipeline.enable_model_cpu_offload()
pipeline.enable_attention_slicing()

if torch.cuda.device_count() > 1:
    pipeline.enable_vae_slicing()
    print("VAE slicing activado (2 GPUs detectadas).")

pipeline.set_progress_bar_config(disable=True)

print(f"Anything V5 configurado com cpu_offload.")
for i in range(torch.cuda.device_count()):
    used  = torch.cuda.memory_allocated(i) // 1024**2
    total = torch.cuda.get_device_properties(i).total_memory // 1024**2
    print(f"GPU {i}: {used} MB / {total} MB")

## 4 - Teste rápido SDXL

In [ ]:
print("A gerar imagem de teste (64x64 steps=1 para ser rápido)...")

test_img = pipeline(
    prompt="anime girl, simple test",
    negative_prompt=NEGATIVE_PROMPT,
    width=64, height=64,
    num_inference_steps=1,
    guidance_scale=1.0,
).images[0]

print(f"Teste OK — tamanho: {test_img.size}")
del test_img

## 5 - Funções auxiliares e store de jobs

In [ ]:
# ── Store de jobs em memória
jobs: Dict[str, Dict[str, Any]] = {}
jobs_lock = threading.Lock()

def create_job(job_id: str) -> str:
    with jobs_lock:
        jobs[job_id] = {
            "status":     "queued",
            "progress":   0,
            "result":     None,
            "error":      None,
            "created_at": datetime.now(),
        }
    return job_id

def update_job(job_id: str, **kwargs) -> None:
    with jobs_lock:
        if job_id in jobs:
            jobs[job_id].update(kwargs)

def cleanup_expired_jobs() -> None:
    cutoff = datetime.now() - timedelta(minutes=JOB_TTL_MINUTES)
    with jobs_lock:
        expired = [jid for jid, j in jobs.items() if j["created_at"] < cutoff]
        for jid in expired:
            jobs.pop(jid, None)
    if expired:
        print(f"[cleanup] {len(expired)} jobs expirados removidos.")

# ── Validação
def require_worker_key(x_worker_key: Optional[str]) -> None:
    if WORKER_API_KEY and x_worker_key != WORKER_API_KEY:
        raise HTTPException(status_code=401, detail="Chave do worker inválida.")

def safe_job_id(job_id: str) -> str:
    cleaned = re.sub(r"[^a-zA-Z0-9_\-]", "_", job_id.strip())
    if not cleaned:
        raise HTTPException(status_code=400, detail="job_id inválido.")
    return cleaned

# ── Geração de imagem em memória
def generate_image_bytes(
    prompt: str,
    width: int = DEFAULT_WIDTH,
    height: int = DEFAULT_HEIGHT,
    steps: int = DEFAULT_STEPS,
    guidance: float = DEFAULT_GUIDANCE,
    seed: Optional[int] = None,
) -> bytes:
    full_prompt = f"{prompt}, {DEFAULT_ANIME_SUFFIX}"
    generator   = torch.Generator(device=SDXL_DEVICE).manual_seed(seed) if seed else None

    with torch.no_grad():
        result = pipeline(
            prompt=full_prompt,
            negative_prompt=NEGATIVE_PROMPT,
            width=width,
            height=height,
            num_inference_steps=steps,
            guidance_scale=guidance,
            generator=generator,
        )

    img = result.images[0]
    buf = io.BytesIO()
    img.save(buf, format="PNG")
    return buf.getvalue()

def image_to_base64(png_bytes: bytes) -> str:
    return base64.b64encode(png_bytes).decode("utf-8")

print("Funções auxiliares e store de jobs definidos.")

## 6 - FastAPI app

In [ ]:
app = FastAPI(title="AOOP ARMONYX Worker 2 — SDXL Image Generator")

# ── Modelos de pedido
class GenerateSceneRequest(BaseModel):
    job_id:       str            = Field(..., min_length=1)
    scene_index:  int            = Field(..., ge=1)
    image_prompt: str            = Field(..., min_length=3)
    width:        int            = Field(DEFAULT_WIDTH,    ge=64,  le=1024)
    height:       int            = Field(DEFAULT_HEIGHT,   ge=64,  le=1024)
    steps:        int            = Field(DEFAULT_STEPS,    ge=1,   le=50)
    guidance:     float          = Field(DEFAULT_GUIDANCE, ge=1.0, le=20.0)
    seed:         Optional[int]  = None

class SceneItem(BaseModel):
    scene_index:  int
    image_prompt: str
    seed:         Optional[int] = None

class GenerateImagesRequest(BaseModel):
    job_id: str             = Field(..., min_length=1)
    scenes: List[SceneItem] = Field(..., min_length=1)
    width:  int             = Field(DEFAULT_WIDTH,  ge=64, le=1024)
    height: int             = Field(DEFAULT_HEIGHT, ge=64, le=1024)
    steps:  int             = Field(DEFAULT_STEPS,  ge=1,  le=50)
    guidance: float         = Field(DEFAULT_GUIDANCE, ge=1.0, le=20.0)

# ── Background tasks
def _task_generate_scene(job_id: str, req: GenerateSceneRequest) -> None:
    cleanup_expired_jobs()
    update_job(job_id, status="running", progress=10)
    try:
        png_bytes = generate_image_bytes(
            prompt=req.image_prompt,
            width=req.width, height=req.height,
            steps=req.steps, guidance=req.guidance,
            seed=req.seed,
        )
        update_job(job_id, status="done", progress=100, result={
            "scenes": [{
                "scene_index":  req.scene_index,
                "image_prompt": req.image_prompt,
                "png_bytes":    png_bytes,
                "size_bytes":   len(png_bytes),
            }]
        })
    except Exception as e:
        update_job(job_id, status="error", error=str(e))

def _task_generate_images(job_id: str, req: GenerateImagesRequest) -> None:
    cleanup_expired_jobs()
    update_job(job_id, status="running", progress=0)
    total  = len(req.scenes)
    scenes_result = []
    try:
        for i, scene in enumerate(req.scenes):
            progress = int((i / total) * 100)
            update_job(job_id, progress=progress)

            png_bytes = generate_image_bytes(
                prompt=scene.image_prompt,
                width=req.width, height=req.height,
                steps=req.steps, guidance=req.guidance,
                seed=scene.seed,
            )
            scenes_result.append({
                "scene_index":  scene.scene_index,
                "image_prompt": scene.image_prompt,
                "png_bytes":    png_bytes,
                "size_bytes":   len(png_bytes),
            })

        update_job(job_id, status="done", progress=100, result={"scenes": scenes_result})
    except Exception as e:
        update_job(job_id, status="error", error=str(e))

# ── Endpoints
@app.get("/health")
def health():
    gpu_info = [{"index": i, "name": torch.cuda.get_device_name(i),
                 "memory_allocated_mb": torch.cuda.memory_allocated(i) // 1024**2,
                 "memory_total_mb": torch.cuda.get_device_properties(i).total_memory // 1024**2}
                for i in range(torch.cuda.device_count())]
    return {
        "status":          "ok",
        "timestamp":       datetime.now().isoformat(),
        "model":           SDXL_MODEL_NAME,
        "model_loaded":    pipeline is not None,
        "sdxl_device":     SDXL_DEVICE,
        "jobs_in_memory":  len(jobs),
        "gpus":            gpu_info,
    }

@app.post("/generate-scene", status_code=202)
def generate_scene(
    req: GenerateSceneRequest,
    background_tasks: BackgroundTasks,
    x_worker_key: Optional[str] = Header(default=None),
):
    require_worker_key(x_worker_key)
    job_id = safe_job_id(req.job_id)
    create_job(job_id)
    background_tasks.add_task(_task_generate_scene, job_id, req)
    return {"job_id": job_id, "status": "queued"}

@app.post("/generate-images", status_code=202)
def generate_images(
    req: GenerateImagesRequest,
    background_tasks: BackgroundTasks,
    x_worker_key: Optional[str] = Header(default=None),
):
    require_worker_key(x_worker_key)
    job_id = safe_job_id(req.job_id)
    create_job(job_id)
    background_tasks.add_task(_task_generate_images, job_id, req)
    return {"job_id": job_id, "status": "queued", "total_scenes": len(req.scenes)}

@app.get("/status/{job_id}")
def get_status(job_id: str, x_worker_key: Optional[str] = Header(default=None)):
    require_worker_key(x_worker_key)
    job = jobs.get(safe_job_id(job_id))
    if not job:
        raise HTTPException(status_code=404, detail="Job não encontrado.")
    result    = job.get("result") or {}
    n_scenes  = len(result.get("scenes", []))
    return {
        "job_id":       job_id,
        "status":       job["status"],
        "progress":     job["progress"],
        "error":        job.get("error"),
        "scenes_ready": n_scenes,
    }

@app.get("/result/{job_id}/scene/{scene_index}")
def result_scene(
    job_id: str,
    scene_index: int,
    x_worker_key: Optional[str] = Header(default=None),
):
    require_worker_key(x_worker_key)
    job = jobs.get(safe_job_id(job_id))
    if not job:
        raise HTTPException(status_code=404, detail="Job não encontrado.")
    if job["status"] != "done":
        raise HTTPException(status_code=409, detail=f"Job ainda não concluído: {job['status']}")

    scenes = job.get("result", {}).get("scenes", [])
    match  = next((s for s in scenes if s["scene_index"] == scene_index), None)
    if not match:
        raise HTTPException(status_code=404, detail=f"Cena {scene_index} não encontrada.")

    return StreamingResponse(
        io.BytesIO(match["png_bytes"]),
        media_type="image/png",
        headers={"Content-Disposition": f"attachment; filename=scene_{scene_index:03d}.png"},
    )

@app.get("/result/{job_id}/images")
def result_images(job_id: str, x_worker_key: Optional[str] = Header(default=None)):
    require_worker_key(x_worker_key)
    job = jobs.get(safe_job_id(job_id))
    if not job:
        raise HTTPException(status_code=404, detail="Job não encontrado.")
    if job["status"] != "done":
        raise HTTPException(status_code=409, detail=f"Job ainda não concluído: {job['status']}")

    scenes = job.get("result", {}).get("scenes", [])
    return {
        "job_id": job_id,
        "total":  len(scenes),
        "scenes": [
            {
                "scene_index":  s["scene_index"],
                "image_prompt": s["image_prompt"],
                "size_bytes":   s["size_bytes"],
                "image_base64": image_to_base64(s["png_bytes"]),
            }
            for s in sorted(scenes, key=lambda x: x["scene_index"])
        ],
    }

print("FastAPI app definida.")

## 7 - Iniciar FastAPI + ngrok

In [ ]:
def start_uvicorn():
    import uvicorn
    uvicorn.run(app, host="0.0.0.0", port=PORT, log_level="info")

ngrok.kill()
time.sleep(1)

server_thread = threading.Thread(target=start_uvicorn, daemon=True)
server_thread.start()
time.sleep(5)

try:
    r = requests.get(f"http://localhost:{PORT}/health", timeout=10)
    r.raise_for_status()
    print("Servidor local iniciado.")
    print(json.dumps(r.json(), indent=2, ensure_ascii=False))
except Exception as e:
    raise RuntimeError(f"Erro ao iniciar FastAPI local: {e}")

conf.get_default().auth_token = NGROK_TOKEN
tunnel       = ngrok.connect(PORT, domain=NGROK_DOMAIN) if NGROK_DOMAIN else ngrok.connect(PORT)
WORKER_2_URL = tunnel.public_url

print("=" * 80)
print("WORKER_2_URL=" + WORKER_2_URL)
print("=" * 80)
for ep in ["/health", "/generate-scene", "/generate-images",
           "/status/{job_id}", "/result/{job_id}/scene/{index}", "/result/{job_id}/images"]:
    print(f"  {WORKER_2_URL}{ep}")

## 8 - Testes rápidos

In [ ]:
import json, requests, time, base64

headers = {"X-Worker-Key": WORKER_API_KEY} if WORKER_API_KEY else {}

# ── Health
r = requests.get(f"{WORKER_2_URL}/health", timeout=20)
print("Health:", r.status_code)
print(json.dumps(r.json(), indent=2, ensure_ascii=False))

## 9 - Keep-alive

In [ ]:
print("Worker 2 activo. Mantém esta célula a correr enquanto usares o backend.")
print("-" * 80)

while True:
    try:
        r    = requests.get(f"http://localhost:{PORT}/health", timeout=10)
        data = r.json()
        gpu_text = [
            f"GPU{g['index']}: {g['memory_allocated_mb']}/{g['memory_total_mb']}MB"
            for g in data.get("gpus", [])
        ]
        print(
            f"[{datetime.now().strftime('%H:%M:%S')}] "
            f"model={data.get('model_loaded')} "
            f"jobs={data.get('jobs_in_memory', 0)} "
            f"| {' | '.join(gpu_text)}"
        )
    except Exception as e:
        print(f"[{datetime.now().strftime('%H:%M:%S')}] ERRO: {e}")
    time.sleep(30)